
<a href="https://colab.research.google.com/github/glenchen0914-glitch/00631L-quant/blob/main/notebooks/00631L_Colab_V4_5.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# 00631L Quant V4.5.1

手機版一鍵執行。會下載 GitHub 專案、連接 Google Drive、執行完整回測並顯示每日決策。


In [ ]:
#@title ① 連接 Google Drive 並執行 V4.5.1
GITHUB_USER = "glenchen0914-glitch"  #@param {type:"string"}
REPO = "00631L-quant"  #@param {type:"string"}

!pip -q install yfinance pandas numpy pyarrow plotly

from google.colab import drive
drive.mount('/content/drive')

import os, shutil, subprocess
from pathlib import Path

os.chdir('/content')
repo_path = Path('/content') / REPO
if repo_path.exists():
    shutil.rmtree(repo_path)

repo_url = f"https://github.com/{GITHUB_USER}/{REPO}.git"
result = subprocess.run(["git", "clone", repo_url], text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError("GitHub 專案下載失敗：\n" + result.stderr)

os.chdir(repo_path)

drive_root = Path('/content/drive/MyDrive/00631L_Quant_V4_5_1')
drive_reports = drive_root / 'reports'
drive_reports.mkdir(parents=True, exist_ok=True)

local_reports = repo_path / 'reports'
if local_reports.is_symlink() or local_reports.exists():
    if local_reports.is_symlink() or local_reports.is_file():
        local_reports.unlink()
    else:
        shutil.rmtree(local_reports)
os.symlink(drive_reports, local_reports, target_is_directory=True)

result = subprocess.run(["python", "run_pipeline.py"], text=True)
if result.returncode != 0:
    raise RuntimeError("V4.5.1 執行失敗，請把紅色錯誤畫面截圖給我。")

print("✅ V4.5.1 完整流程執行完成")

In [ ]:

#@title ② 顯示手機版決策卡
from IPython.display import HTML, display
from pathlib import Path
html = Path('reports/dashboard.html').read_text(encoding='utf-8')
display(HTML(html))


In [ ]:

#@title ③ 顯示策略排行榜與波段追蹤表
import pandas as pd
display(pd.read_csv('reports/strategy_leaderboard.csv').head(10))
display(pd.read_csv('reports/wave_tracking.csv').tail(20))
